In [1]:
import numpy as np
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "stsb"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1379
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 5749
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1379
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'A plane is taking off.',
 'sentence2': 'An air plane is taking off.',
 'label': 5.0,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1']

'Some men are fighting.'

In [11]:
raw_train_dataset[5]['sentence2']

'Two men are fighting.'

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': Value('float32'),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5749
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1379
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 66]),
 'token_type_ids': torch.Size([32, 66]),
 'attention_mask': torch.Size([32, 66])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
num_labels = 1

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    problem_type="regression",
    # id2label=id2label,
    # label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(4.9161, grad_fn=<MseLossBackward0>) torch.Size([32, 1])


In [25]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda')

### Load Trained Weights

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
from train_eval_func import eval_loop

In [28]:
eval_loop(model, val_dataloader, task_name, device, regression=True)[0]

{'pearson': 0.900084701022039, 'spearmanr': 0.8962584593578651}

# 5. Merge Most Similar Layers

In [29]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from tqdm import tqdm

In [30]:
import os
from merge_helper import merge_bert_layers, LayerMergeTracker
from train_eval_func import train, get_primary_metric, set_lr_scheduler, iterative_cka_merge_and_train, get_recovery_epoch
from torch.optim import AdamW

In [31]:
init_metric = eval_loop(model, val_dataloader, task_name, device, regression=True)[0]
init_metric

{'pearson': 0.900084701022039, 'spearmanr': 0.8962584593578651}

In [32]:
tracker = LayerMergeTracker(num_layers=12)

print("Initial state:")
print(f"  {tracker.get_mapping()}")
print(f"  Total layers: {len(tracker)}")
print()

Initial state:
  [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11]]
  Total layers: 12



In [33]:
from CKA import CKAEvaluator

In [34]:
cka_eval = CKAEvaluator(device)

In [35]:
recovery_epochs = get_recovery_epoch(task_name)
recovery_epochs

10

In [36]:
history = iterative_cka_merge_and_train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    task_name='stsb',
    device=device,
    cka_evaluator=cka_eval,
    tracker=tracker,
    init_metric=init_metric,
    num_merges=6,
    target_layers=[6, 7, 8],  # Save models with 6, 7, 8 layers
    recovery_epochs=recovery_epochs,
    recovery_lr=1e-5,
    save_dir='./weights/',
    regression=True
)

Target Metric for stsb: pearson
Original pearson score: 0.9001
Recovery threshold: 0.0090

Merge Iteration: 1/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|██████████| 180/180 [01:39<00:00,  1.81it/s]


  Similarity Stats:
    Average: 0.924413
    Highest: 0.993546
    Lowest:  0.545679

[2/5] Merging Layers...
  Merged Layers: 8 + 9
  Layer Composition: [[0], [1], [2], [3], [4], [5], [6], [7], [8, 9], [10], [11]]
  Total Layers: 11

[3/5] Evaluating Post-Merge Performance...
  Metrics:
    ★ pearson: 0.8950
      spearmanr: 0.8925

[4/5] Recovery Training...
  Performance drop: 0.0051 (threshold: 0.0090)
  → Recovery training NOT needed

[5/5] Saving Model...
  • 11 layers (not in target list, skipping save)

Merge Iteration: 2/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|██████████| 180/180 [01:11<00:00,  2.51it/s]


  Similarity Stats:
    Average: 0.915361
    Highest: 0.990614
    Lowest:  0.549616

[2/5] Merging Layers...
  Merged Layers: 6 + 7
  Layer Composition: [[0], [1], [2], [3], [4], [5], [6, 7], [8, 9], [10], [11]]
  Total Layers: 10

[3/5] Evaluating Post-Merge Performance...
  Metrics:
    ★ pearson: 0.8905
      spearmanr: 0.8884

[4/5] Recovery Training...
  Performance drop: 0.0096 (threshold: 0.0090)
  → Recovery training NEEDED
  → Training: 10 epochs, lr=1e-05


...

Training:  50%|█████     | 900/1800 [04:28<03:46,  3.97it/s]  


<----------------- Epoch 5 ----------------->
Loss: 0.05, Training Metrics:
  pearson: 0.9881
  spearmanr: 0.9862
Validation Loss: 0.48, Validation Metrics:
  pearson: 0.8909
  spearmanr: 0.8865
Elapsed Time: 48.0331 sec


Training:  60%|██████    | 1080/1800 [05:17<03:31,  3.40it/s]


<----------------- Epoch 6 ----------------->
Loss: 0.05, Training Metrics:
  pearson: 0.9893
  spearmanr: 0.9875
Validation Loss: 0.51, Validation Metrics:
  pearson: 0.8901
  spearmanr: 0.8855
Elapsed Time: 45.1638 sec
>>>>> Early stopping callback <<<<<

Total Training Time: 317.55 sec
Chosen Primary Metric: pearson
Best Metric Run: 0.8913

  → Loading best checkpoint...


  ✓ Loaded best checkpoint from epoch 4
    Best val loss: 0.4983
    Best val metrics: {'pearson': 0.8913027778969957, 'spearmanr': 0.8872237875495089}
  ✓ Cleaned up temporary checkpoint

[5/5] Saving Model...
  ✓ Saved 6-layer model to ./weights/layer-6-stsb.pt

MERGE LOOP COMPLETED
Total merges: 6
Final layers: 6
Final composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]


In [37]:
# See what happened
print("Merge history:")
for i, iteration in enumerate(history['iterations']):
    print(f"  Iteration {iteration['iteration']}:")
    print(f"    Merged: {iteration['merged_layers']}")
    print(f"    Layers after: {iteration['num_layers_after']}")
    print(f"    Composition: {history['layer_compositions'][i]}")
    
    if history['training_stats'][i]:
        final_metrics = history['metrics_after_recovery'][i]
        print(f"    Final metrics: {final_metrics}")

Merge history:
  Iteration 1:
    Merged: (8, 9)
    Layers after: 11
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]
  Iteration 2:
    Merged: (6, 7)
    Layers after: 10
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]
    Final metrics: {'pearson': 0.8962231668857096, 'spearmanr': 0.8921232625576194}
  Iteration 3:
    Merged: (7, 8)
    Layers after: 9
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]
    Final metrics: {'pearson': 0.8927645647315687, 'spearmanr': 0.8892170950741984}
  Iteration 4:
    Merged: (5, 6)
    Layers after: 8
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]
    Final metrics: {'pearson': 0.8925750776984478, 'spearmanr': 0.8882112740962673}
  Iteration 5:
    Merged: (1, 2)
    Layers after: 7
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7, 8, 9, 10], [11]]
    Final metrics: {'pearson': 0.8941012834723292, 'spearmanr': 0.8903806368014037}
  Iteration 6:
    Merged: (4, 5)
   